In [1]:
%pip install -q langchain langchain-community langchain-google-genai langchain-text-splitters pymupdf faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [3]:
from tkinter import Tk
from tkinter.filedialog import askopenfilename

Tk().withdraw()  # Hide root window

file_path = askopenfilename(title="Select a file")

print("Selected file:", file_path)

Selected file: /Users/gagandeep/Documents/Other People Resume/deepanshu_sde (1).pdf


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)

import os


/var/folders/01/yby9_s_j5156kjz10qb2rlzc0000gn/T/ipykernel_11571/1647180047.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [6]:
# Step 1: Load Documents
# loader = PyMuPDFLoader("/application/deepanshu_sde(1).pdf")
loader = PyMuPDFLoader("/Users/gagandeep/Documents/Other People Resume/deepanshu_sde (1).pdf")
docs = loader.load()

In [7]:
# Step 2: Split Documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)


In [8]:
# Step 3: Generate Embeddings
from getpass import getpass
os.environ["GOOGLE_API_KEY"] = getpass("Enter API Key: ")

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"   # was "models/embedding-001"
)

test = embeddings.embed_query("hello")
print(len(test))   # should print 3072 by default



3072


In [9]:
# Step 4: Create and Save the Database
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)


In [10]:
# Step 5: Create Retriever
retriever = vectorstore.as_retriever()


In [11]:
# Step 6: Create Prompt
prompt = PromptTemplate.from_template(
    """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.

#Context:
{context}

#Question:
{question}

#Answer:"""
)



In [12]:
# Step 7: Load LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)  # `model=`, not `model_name=`

In [13]:
# Step 8: Create Chain
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [14]:
# Try it
answer = chain.invoke("What is this document about?")
print(answer)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 33.328059418s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url